# 23_security_reliability: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/23_security_reliability/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Authenticate a trusted identity and demonstrate a hostile filter boundary.
Run: uv run python modules/23_security_reliability/examples/01_from_scratch.py
CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
Read the module's setup for optional extras and server requirements.
"""

import json
import tempfile
from pathlib import Path

from common.ops.tenancy import tenant_filter
from common.security.audit import AuditLog
from common.security.identity import Keyring, Principal


def main():
    principal = Principal("reader-a", "seller-a", "reader")
    token = "local-test-token-0123456789"
    keys = Keyring({token: principal})
    assert keys.authenticate(token) == principal
    assert principal.permits("read") and not principal.permits("write")
    hostile = {"op": "or", "conditions": [{"op": "eq", "field": "seller_id", "value": "seller-b"}]}
    print(tenant_filter(principal, hostile))
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary) / "audit.jsonl"
        audit = AuditLog(path)
        audit.record(
            subject=principal.subject,
            tenant=principal.tenant,
            action="/search",
            outcome="200",
            request_id="demo",
        )
        event = json.loads(path.read_text())
        assert token not in path.read_text() and "query" not in event
        print(event)
    print("Audit records identity and outcome, never request bodies or embeddings.")


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/23_security_reliability/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None or "qdrant-secure" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra qdrant; start qdrant-secure and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Verify native database TLS, credentials and the danger of omitted filters.
    Run: uv run python modules/23_security_reliability/examples/02_production.py
    CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
    Read the module's setup for optional extras and server requirements.
    """
    # course-extra: qdrant
    # course-profile: qdrant-secure

    from common.ops.native import native_security

    def main():
        result = native_security()
        assert result["scoped_ids"] == [1]
        assert set(result["omitted_filter_ids"]) == {1, 2}
        print(result)
        print("A database-wide read key is not a seller identity. The API must own scope.")

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/23_security_reliability/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("fastapi") is None:
    print("SKIP native comparison: --extra apps")
else:
    """Verify authenticated seller requests and persistent deletion in the real API.
    Run: uv run python modules/23_security_reliability/examples/03_measure.py
    CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
    Read the module's setup for optional extras and server requirements.
    """
    # course-extra: apps

    import tempfile
    from pathlib import Path

    from fastapi.testclient import TestClient

    from common.apps.secure_api import create_app
    from common.security.identity import Keyring, Principal

    def main():
        token = "local-demo-admin-0123456789"
        ring = Keyring({token: Principal("admin", "seller-000", "admin")})
        headers = {"Authorization": "Bearer " + token}
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)

            def application():
                return create_app(keyring=ring, root=root, backend="tinyvec", limit=500)

            with TestClient(application()) as client:
                assert client.post("/search", json={}).status_code == 401
                rows = client.post("/search", json={}, headers=headers).json()["data"]["hits"]
                assert rows and all(r["metadata"]["seller_id"] == "seller-000" for r in rows)
                record_id = rows[0]["id"]
                assert client.post("/ops/backup", headers=headers).status_code == 200
                assert client.delete("/listings/" + record_id, headers=headers).status_code == 200
            with TestClient(application()) as client:
                rows = client.post("/search", json={}, headers=headers).json()["data"]["hits"]
                assert record_id not in [r["id"] for r in rows]
        print("No cross-tenant rows, unauthorized request rejected, deletion survives restart.")

    if __name__ == "__main__":
        main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.security.identity import Principal

p = Principal("reader", "seller-a", "reader")
print(p.permits("read"), p.permits("write"))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.